# Prompt Caching

Prompt caching speeds up responses and cuts cost by **reusing the preprocessing work** from earlier requests instead of throwing it away.

> Concept lesson — no code. The `cache_control` syntax comes next in **Rules of prompt caching**, then **Prompt caching in action**.

## What normally happens

Before generating a single token, Claude does heavy preprocessing on your input:

1. **Tokenize** the prompt into pieces
2. **Embed** each token
3. **Add context** based on surrounding text
4. *Then* generate the output

After responding, **all of that work is discarded**. Send overlapping content again — say, iterating on a summary of the *same* long document — and Claude redoes the identical preprocessing from scratch every time.

## How caching fixes it

On the first request, Claude does the usual preprocessing but **stores** the result in a cache keyed to that content. On later requests, any prefix that matches is **read from the cache** instead of recomputed.

- **First request → cache write** (a bit more expensive)
- **Later requests → cache read** (much cheaper + faster)

It only helps for the **stable prefix** of your prompt — caching is prefix-based, so the cached content must sit at the *front* and stay byte-identical across requests.

## Benefits & limits

| Benefits | Limits |
|----------|--------|
| Faster responses on cache hits | Only helps when you resend the **same** content |
| Lower cost on the cached portion | Cache entries **expire** (see TTL note) |
| Automatic: first request writes, rest read | Most valuable when that content recurs **frequently** |

**Great fits:** document-analysis workflows (many questions about one big document), iterative editing where a large base stays constant while you tweak specifics, long system prompts / tool definitions reused across calls.

## Accuracy note — cache lifetime

The course says the cache lives **one hour**. On current models the default is a **5-minute** TTL that **refreshes on every hit** (so steady traffic keeps it warm), with an optional **1-hour** TTL you request explicitly. Roughly: cache **reads** cost ~10% of normal input tokens; cache **writes** cost ~25% more (5-min) or more for the 1-hour tier. We'll confirm the exact `cache_control` fields against the docs in the next lesson.

> **Codebase aside:** for a code assistant, the natural cache target is the large, stable context you resend on every question — a big system prompt, tool schemas, or a fixed set of retrieved files — so repeated questions about the same code only pay full price once.

Next: **Rules of prompt caching** — where to place `cache_control`, prefix rules, and what invalidates a cache.